<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 1. כותרת ואזהרת סימולציה


        **סימולציה לימודית בלבד.**

        אין חיבור לבנק, לכרטיס אשראי או לספק תשלומים אמיתי.
        אין להשתמש בנתונים פיננסיים אמיתיים.

        הלוגיקה הכספית דטרמיניסטית. שכבת ה־LLM אופציונלית ומייעצת בלבד.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 2. מבנה ההגשה ומטרות הפרויקט


        ההגשה כוללת את **המאגר המלא** ואת המחברת.

        - המימוש המלא נמצא תחת `src/agentic_payments/`.
        - המחברת היא הסבר והרצה של הקוד האמיתי מתוך המאגר.
        - אין במחברת העתק של קוד הייצור ואין שחזור של המאגר.

        מטרות הפרויקט הן תשלומים בטוחים בסביבה מקומית, סוכנים בעלי גבולות ברורים,
        התמדה אטומית, עקביות במקביליות והרצה מלאה ללא מפתח API.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 3. מיפוי דרישות המרצה


        | דרישה | רכיב בפרויקט | הדגמה |
        |---|---|---|
        | תוצאה משותפת | `AgentResult` | הצגת קוד המקור בפועל |
        | ניתוב ותזמור | `RouterAgent`, `OrchestratorAgent` | עשרת התרחישים |
        | כלים וזיכרון | `PaymentToolRegistry`, `BusinessMemory` | מיפוי וכלי הסבר |
        | בדיקות סיכון | `FraudDetectionAgent`, `SecurityAgent` | עסקה בסיכון `HIGH` |
        | בקרת איכות | `CriticAgent`, `FallbackAgent` | כל תוצאת תזמור |
        | רכיבים מתקדמים | `PolicyAgent`, `ReflectionAgent` | דוגמאות ייעודיות |
        | בטיחות | Locks, Idempotency, Audit Outbox | התמדה ומקביליות |

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 4. ארכיטקטורה וגבולות אחריות


        ```text
        User → Router → Orchestrator → ToolGuardrails → PaymentFacade
                                                       ↓
                                              PaymentDomainService
                                                       ↓
                                      Locks → Unit of Work → JSON state
                                                       ↓
                                  Fraud / Security / Memory / Audit Outbox
        ```

        רק `PaymentDomainService` רשאי לשנות מצב כספי.
        הסוכנים מקבלים עובדות בלתי־משתנות ואינם משנים יתרות.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 5. הכנת סביבת הרצה מתוך המאגר


        המחברת מיועדת להרצה משורש המאגר.
        תא ההכנה בודק את `src/agentic_payments/`, מוסיף את `src` ל־`sys.path`
        ויוצר את כל קובצי ההדגמה בתיקייה זמנית של מערכת ההפעלה.

        אם קוד המקור אינו קיים, מתקבלת הודעה ברורה במקום הורדה מהרשת.

</div>

In [ ]:
import asyncio
import inspect
import sys
import tempfile
from datetime import UTC, datetime
from decimal import Decimal
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
SOURCE_ROOT = REPOSITORY_ROOT / "src"
if not (SOURCE_ROOT / "agentic_payments").is_dir():
    raise RuntimeError(
        "Run this notebook from the root of the submitted repository."
    )
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

NOTEBOOK_RUNTIME = tempfile.TemporaryDirectory(
    prefix="agentic-payment-notebook-"
)
RUNTIME_ROOT = Path(NOTEBOOK_RUNTIME.name)
DATA_ROOT = REPOSITORY_ROOT / "data"

def data_snapshot():
    return {
        path.relative_to(DATA_ROOT).as_posix(): path.read_bytes()
        for path in DATA_ROOT.rglob("*")
        if path.is_file()
    }

REPOSITORY_DATA_BEFORE = data_snapshot()

from agentic_payments.agents import PolicyAgent
from agentic_payments.application import AgentResult
from agentic_payments.bootstrap import build_application
from agentic_payments.domain import Intent
from agentic_payments.infrastructure import Settings
from agentic_payments.infrastructure.llm.sdk_guardrails import (
    _validate_tool_output,
)
from agentic_payments.tools.payment_tools import _TOOL_NAMES

DEMO_TIME = datetime(2026, 2, 1, 12, 0, tzinfo=UTC)

def demo_phone(seed):
    return "050" + f"{seed:07d}"[-7:]

async def isolated_container(label, **overrides):
    root = RUNTIME_ROOT / label
    root.mkdir(parents=True, exist_ok=True)
    values = {
        "app_env": "test",
        "llm_provider": "rule_based",
        "enable_llm_router": False,
        "state_file": root / "payment_state.json",
        "audit_file": root / "audit_log.jsonl",
    }
    values.update(overrides)
    settings = Settings(_env_file=None, **values)
    return await build_application(settings), settings

async def create_demo_user(container, name, seed, balance, key):
    return await container.orchestrator.handle(
        (
            f'createUser name="{name}" phone={demo_phone(seed)} '
            f"initial_balance={balance}"
        ),
        correlation_id=f"COR-{key}",
        idempotency_key=f"IDEM-{key}",
        requested_at=DEMO_TIME,
    )

print("Repository package import success: yes")
print("Repository-based source of truth: src/agentic_payments")
print("Temporary runtime isolation: yes")


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 6. המבנה המשותף AgentResult


        כל סוכן מחזיר `AgentResult` באותו מבנה.
        התא הבא מציג את ההצהרה האמיתית מתוך `src`, באמצעות `inspect.getsource()`.

</div>

In [2]:
agent_result_source = inspect.getsource(AgentResult)
assert "agent_name" in agent_result_source
assert "output" in agent_result_source
assert "confidence" in agent_result_source
assert "metadata" in agent_result_source
print(agent_result_source)


@dataclass(slots=True)
class AgentResult:
    """Common result returned by every agent."""

    agent_name: str
    output: Any
    confidence: float = 1.0
    metadata: Optional[Dict[str, Any]] = None  # noqa: UP006,UP045 - required shape.

    def __post_init__(self) -> None:
        _text(self.agent_name, "agent_name")
        if isinstance(self.confidence, bool) or not isinstance(self.confidence, (int, float)):
            raise TypeError("confidence must be numeric and not bool")
        self.confidence = float(self.confidence)
        if not 0.0 <= self.confidence <= 1.0:
            raise ValueError("confidence must be between 0 and 1")
        if self.metadata is not None:
            if not isinstance(self.metadata, dict):
                raise TypeError("metadata must be a dictionary or None")
            self.metadata = dict(self.metadata)



<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 7. מודל הדומיין והסוכנים


        | רכיב | אחריות |
        |---|---|
        | `User` | זהות משתמש וטלפון מנורמל |
        | `Wallet` | יתרה בלתי־משתנה וגרסה עולה |
        | `Transaction` | העברה, סטטוס והערכת סיכון |
        | `PaymentRequest` | בקשת תשלום ומעבר מצב חד־כיווני |
        | `TransferPolicy` | סכום חיובי, מגבלות והיקף יומי |

        | סוכן | אחריות |
        |---|---|
        | `RouterAgent` | סיווג וחילוץ פרמטרים |
        | `OrchestratorAgent` | תזמור בקשה אחת וכלי ראשי אחד |
        | `FraudDetectionAgent` | ציון סיכון דטרמיניסטי |
        | `SecurityAgent` | בדיקת עקביות לקריאה בלבד |
        | `ExplanationAgent` | הסבר מעובדות הזיכרון |
        | `CriticAgent` | בדיקת איכות התוצאה |
        | `PolicyAgent` | עצת מדיניות לפני פעולה |
        | `ReflectionAgent` | התאוששות בטוחה משגיאה |
        | `FallbackAgent` | חסימת בקשה לא ברורה |

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 8. כלים, Guardrails וזיכרון עסקי


        מיפוי הכלים קבוע: לכל intent נתמך יש כלי אחד.
        `BusinessMemory` שומר את הכוונה, המשתמש, העסקה, הבקשה והתוצאה האחרונה.

        ה־Guardrails דוחים כסף שמקורו ב־float, סודות, הוראות ביצוע ומספרי טלפון מלאים.
        תאריך ISO תקין ומודע לאזור זמן מותר רק בשדה זמן מאושר.

</div>

In [3]:
tool_rows = [
    (intent.value, _TOOL_NAMES[intent])
    for intent in Intent
    if intent in _TOOL_NAMES
]
assert len(tool_rows) == 10
print("Intent-to-tool mapping:")
for intent_name, tool_name in tool_rows:
    print(f"  {intent_name:20} -> {tool_name}")

_validate_tool_output(
    {"requested_at": DEMO_TIME.isoformat(), "facts": {"status": "safe"}}
)
phone_rejected = False
try:
    _validate_tool_output(
        {"facts": {"contact": demo_phone(9876543)}}
    )
except ValueError:
    phone_rejected = True
assert phone_rejected
print("Guardrail aware datetime: PASS")
print("Guardrail complete-phone rejection: PASS")


Intent-to-tool mapping:
  createUser           -> create_user_tool
  checkBalance         -> check_balance_tool
  transferMoney        -> transfer_money_tool
  requestPayment       -> request_payment_tool
  approvePayment       -> approve_payment_tool
  rejectPayment        -> reject_payment_tool
  showTransactions     -> show_transactions_tool
  fraudCheck           -> fraud_check_tool
  securityReview       -> security_review_tool
  explainLastAction    -> explain_last_action_tool
Guardrail aware datetime: PASS
Guardrail complete-phone rejection: PASS


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 9. בניית היישום


        `build_application()` טוען מצב, יוצר מנהל נעילות ועסקאות, בונה שירות דומיין,
        סוכנים, כלים, זיכרון ו־Audit Outbox.

        ההגדרה כאן היא `rule_based`; אין מפתח API ואין אובייקט ספק.

</div>

In [4]:
demo_container, demo_settings = await isolated_container("basic")
assert demo_settings.llm_api_key is None
assert demo_container.llm_runtime is None
print("Application bootstrap: PASS")
print("Provider client constructed: no")


Application bootstrap: PASS
Provider client constructed: no


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 10. הדגמה בסיסית


        הדגמה קצרה מפעילה את ה־`OrchestratorAgent` האמיתי:
        יצירת שני משתמשים, בדיקת יתרה, העברה והצגת היסטוריה.
        הפלט מציג רק סיכום ואינו מציג טלפונים או metadata מלא.

</div>

In [5]:
basic_alice = await create_demo_user(
    demo_container, "Basic Alice", 101, "1000.00", "BASIC-A"
)
basic_bob = await create_demo_user(
    demo_container, "Basic Bob", 102, "200.00", "BASIC-B"
)
basic_alice_id = basic_alice.output["user_id"]
basic_bob_id = basic_bob.output["user_id"]
basic_balance = await demo_container.orchestrator.handle(
    f"checkBalance user_id={basic_alice_id}",
    requested_at=DEMO_TIME,
)
basic_transfer = await demo_container.orchestrator.handle(
    (
        f"transferMoney sender_id={basic_alice_id} "
        f"receiver_id={basic_bob_id} amount=125.00"
    ),
    idempotency_key="IDEM-BASIC-TRANSFER",
    requested_at=DEMO_TIME,
)
basic_history = await demo_container.orchestrator.handle(
    f"showTransactions user_id={basic_alice_id}",
    requested_at=DEMO_TIME,
)
assert basic_balance.output["balance"] == "1000.00"
assert basic_transfer.output["snapshot"]["sender_balance_after"] == "875.00"
assert len(basic_history.output["transactions"]) == 1
print("Basic demo: balance 1000.00 -> 875.00")
print("Basic demo: one persisted transaction")


Basic demo: balance 1000.00 -> 875.00
Basic demo: one persisted transaction


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 11. עשרת תרחישי המרצה


        כל תרחיש מפעיל את זרימת ה־production.
        כשל עסקי צפוי חוזר כ־`ReflectionAgent`; חריגה לא צפויה אינה מוסתרת.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

### תרחיש 1 — יצירת שני משתמשים ובדיקת יתרות

הרצה דרך `OrchestratorAgent` עם assertions.

### תרחיש 2 — העברת כסף מוצלחת

הרצה דרך `OrchestratorAgent` עם assertions.

</div>

In [6]:
scenario_container, scenario_settings = await isolated_container("lecturer")
scenario_alice = await create_demo_user(
    scenario_container, "Scenario Alice", 201, "1000.00", "S1-A"
)
scenario_bob = await create_demo_user(
    scenario_container, "Scenario Bob", 202, "200.00", "S1-B"
)
scenario_alice_id = scenario_alice.output["user_id"]
scenario_bob_id = scenario_bob.output["user_id"]
balance_a = await scenario_container.orchestrator.handle(
    f"checkBalance user_id={scenario_alice_id}", requested_at=DEMO_TIME
)
balance_b = await scenario_container.orchestrator.handle(
    f"checkBalance user_id={scenario_bob_id}", requested_at=DEMO_TIME
)
assert balance_a.output["balance"] == "1000.00"
assert balance_b.output["balance"] == "200.00"
print("PASS — create two users and check balances")

scenario_transfer = await scenario_container.orchestrator.handle(
    (
        f"transferMoney sender_id={scenario_alice_id} "
        f"receiver_id={scenario_bob_id} amount=125.00"
    ),
    idempotency_key="IDEM-S2",
    requested_at=DEMO_TIME,
)
assert scenario_transfer.output["snapshot"]["sender_balance_after"] == "875.00"
assert scenario_transfer.output["snapshot"]["receiver_balance_after"] == "325.00"
print("PASS — successful transfer")


PASS — create two users and check balances
PASS — successful transfer


<div dir="rtl" style="text-align: right; line-height: 1.7;">

### תרחיש 3 — סכום העברה שלילי

הרצה דרך `OrchestratorAgent` עם assertions.

### תרחיש 4 — יתרה לא מספקת

הרצה דרך `OrchestratorAgent` עם assertions.

</div>

In [7]:
negative = await scenario_container.orchestrator.handle(
    (
        f"transferMoney sender_id={scenario_alice_id} "
        f"receiver_id={scenario_bob_id} amount=-1.00"
    ),
    idempotency_key="IDEM-S3",
    requested_at=DEMO_TIME,
)
assert negative.agent_name == "ReflectionAgent"
assert negative.output.error_code == "value_error"
print("PASS — negative transfer amount")

insufficient = await scenario_container.orchestrator.handle(
    (
        f"transferMoney sender_id={scenario_alice_id} "
        f"receiver_id={scenario_bob_id} amount=2000.00"
    ),
    idempotency_key="IDEM-S4",
    requested_at=DEMO_TIME,
)
assert insufficient.agent_name == "ReflectionAgent"
assert insufficient.output.error_code == "insufficient_funds"
print("PASS — insufficient funds")


PASS — negative transfer amount
PASS — insufficient funds


<div dir="rtl" style="text-align: right; line-height: 1.7;">

### תרחיש 5 — מקבל שאינו קיים

הרצה דרך `OrchestratorAgent` עם assertions.

### תרחיש 6 — העברה עצמית

הרצה דרך `OrchestratorAgent` עם assertions.

</div>

In [8]:
missing_receiver = await scenario_container.orchestrator.handle(
    (
        f"transferMoney sender_id={scenario_alice_id} "
        "receiver_id=USR-MISSING amount=1.00"
    ),
    idempotency_key="IDEM-S5",
    requested_at=DEMO_TIME,
)
assert missing_receiver.agent_name == "ReflectionAgent"
assert missing_receiver.output.error_code == "user_not_found"
print("PASS — nonexistent receiver")

self_transfer = await scenario_container.orchestrator.handle(
    (
        f"transferMoney sender_id={scenario_alice_id} "
        f"receiver_id={scenario_alice_id} amount=1.00"
    ),
    idempotency_key="IDEM-S6",
    requested_at=DEMO_TIME,
)
assert self_transfer.agent_name == "ReflectionAgent"
assert self_transfer.output.error_code == "value_error"
print("PASS — self transfer")


PASS — nonexistent receiver
PASS — self transfer


<div dir="rtl" style="text-align: right; line-height: 1.7;">

### תרחיש 7 — יצירה ואישור של בקשת תשלום

הרצה דרך `OrchestratorAgent` עם assertions.

### תרחיש 8 — אישור חוזר של בקשה שהוכרעה

הרצה דרך `OrchestratorAgent` עם assertions.

</div>

In [9]:
payment_request = await scenario_container.orchestrator.handle(
    (
        f"requestPayment requester_id={scenario_bob_id} "
        f"payer_id={scenario_alice_id} amount=30.00"
    ),
    idempotency_key="IDEM-S7-REQUEST",
    requested_at=DEMO_TIME,
)
scenario_request_id = payment_request.output["payment_request_id"]
approved_request = await scenario_container.orchestrator.handle(
    f"approvePayment request_id={scenario_request_id}",
    idempotency_key="IDEM-S7-APPROVE",
    requested_at=DEMO_TIME,
)
assert approved_request.output["payment_request"]["status"] == "APPROVED"
print("PASS — create and approve payment request")

resolved_again = await scenario_container.orchestrator.handle(
    f"approvePayment request_id={scenario_request_id}",
    idempotency_key="IDEM-S8-DIFFERENT",
    requested_at=DEMO_TIME,
)
assert resolved_again.agent_name == "ReflectionAgent"
assert resolved_again.output.error_code == "payment_request_already_resolved"
print("PASS — approve already resolved payment request")


PASS — create and approve payment request


PASS — approve already resolved payment request


<div dir="rtl" style="text-align: right; line-height: 1.7;">

### תרחיש 9 — זיהוי עסקה חשודה

הרצה דרך `OrchestratorAgent` עם assertions.

### תרחיש 10 — הסבר הפעולה האחרונה מזיכרון עסקי

הרצה דרך `OrchestratorAgent` עם assertions.

</div>

In [10]:
high_container, high_settings = await isolated_container(
    "high-risk",
    maximum_single_transfer=Decimal("5000.00"),
    maximum_daily_transfer=Decimal("10000.00"),
)
high_source = await create_demo_user(
    high_container, "High Source", 301, "5000.00", "S9-SOURCE"
)
high_target = await create_demo_user(
    high_container, "High Target", 302, "0.00", "S9-TARGET"
)
high_source_id = high_source.output["user_id"]
high_target_id = high_target.output["user_id"]
high_result = await high_container.orchestrator.handle(
    (
        f"transferMoney sender_id={high_source_id} "
        f"receiver_id={high_target_id} amount=4000.00"
    ),
    idempotency_key="IDEM-S9-RISK",
    requested_at=DEMO_TIME,
)
high_tx_id = high_result.output["transaction_id"]
high_assessment = high_result.output["fraud_assessment"]
assert high_assessment["risk_level"] == "HIGH"
assert high_result.output["snapshot"]["transaction"]["status"] == "FLAGGED"
assert high_result.output["security_review"] is not None
print("PASS — suspicious transaction detection")

high_restart = await build_application(high_settings)
assert high_restart.memory_service.snapshot().last_transaction_id == high_tx_id
explanation = await high_restart.orchestrator.handle(
    "explainLastAction",
    requested_at=DEMO_TIME,
)
assert explanation.output["facts"]["output"]["transaction_id"] == high_tx_id
print("PASS — explain last action using persisted BusinessMemory")


PASS — suspicious transaction detection
PASS — explain last action using persisted BusinessMemory


In [11]:
LECTURER_SCENARIOS_PASSED = True
print("10/10 lecturer scenarios passed")


10/10 lecturer scenarios passed


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 12. עסקה בסיכון גבוה


        העברה של 4000 מתוך 5000 עוברת את ספי הכמות ויחס היתרה.
        `FraudDetectionAgent` מסמן `HIGH`; לאחר מכן `SecurityAgent` בודק את העובדות.

</div>

In [12]:
assert high_assessment["risk_score"] >= 60
assert high_assessment["risk_level"] == "HIGH"
assert high_result.output["snapshot"]["transaction"]["status"] == "FLAGGED"
assert high_result.output["security_review"]["approved"] is True
print(
    "HIGH-risk demo:",
    f"score={high_assessment['risk_score']}",
    "level=HIGH",
    "status=FLAGGED",
)


HIGH-risk demo: score=60 level=HIGH status=FLAGGED


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 13. רכיבים מתקדמים והתמדה


        `PolicyAgent` נותן עצה לפני פעולה.
        `ReflectionAgent` ממיר שגיאת דומיין להנחיית התאוששות בלי לבצע retry.

        מצב JSON, זיכרון עסקי, idempotency ואירועי Audit Outbox נשמרים בתיקייה הזמנית.

</div>

In [13]:
policy_agent = PolicyAgent(
    transfer_policy=high_settings.build_transfer_policy()
)
policy_review = await policy_agent.evaluate_transfer(
    sender_id=high_source_id,
    amount=Decimal("6000.00"),
    balance_before=Decimal("10000.00"),
    previous_transactions=(),
    now=DEMO_TIME,
)
assert policy_review.output.approved is False
assert "policy_violation" in policy_review.output.violations
assert insufficient.agent_name == "ReflectionAgent"
assert insufficient.output.recovery_steps
print("PolicyAgent over-limit rejection: PASS")
print("ReflectionAgent safe recovery advice: PASS")


PolicyAgent over-limit rejection: PASS
ReflectionAgent safe recovery advice: PASS


In [14]:
restarted_state = high_restart.snapshot()
assert len(restarted_state.users) == 2
assert high_tx_id in restarted_state.transactions
assert restarted_state.memory.last_transaction_id == high_tx_id
PERSISTENCE_PASSED = True
print("PASS — JSON persistence and BusinessMemory survived restart")

outbox_result = await high_restart.flush_outbox()
audit_events = await high_restart.audit_repository.list_all()
assert outbox_result.pending_after == 0
assert audit_events
assert not high_restart.snapshot().pending_audit_events
OUTBOX_PASSED = True
print("PASS — Audit Outbox delivered all temporary audit events")


PASS — JSON persistence and BusinessMemory survived restart
PASS — Audit Outbox delivered all temporary audit events


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 14. בטיחות במקביליות


        ההדגמה מריצה שלושה מצבים אמיתיים:

        1. שתי משיכות מתחרות מאותה יתרה.
        2. שתי הפקדות מתחרות לאותו ארנק.
        3. העברות בכיוונים מנוגדים תחת timeout.

        סדר נעילות דטרמיניסטי מונע deadlock.
        ההבטחה היא לתהליך Python ול־event loop יחידים.

</div>

In [15]:
withdraw_container, _ = await isolated_container("concurrent-withdraw")
withdraw_source = await create_demo_user(
    withdraw_container, "Withdraw Source", 401, "100.00", "CW-S"
)
withdraw_a = await create_demo_user(
    withdraw_container, "Withdraw A", 402, "0.00", "CW-A"
)
withdraw_b = await create_demo_user(
    withdraw_container, "Withdraw B", 403, "0.00", "CW-B"
)
source_id = withdraw_source.output["user_id"]
withdraw_results = await asyncio.gather(
    withdraw_container.orchestrator.handle(
        (
            f"transferMoney sender_id={source_id} "
            f"receiver_id={withdraw_a.output['user_id']} amount=80.00"
        ),
        idempotency_key="IDEM-CW-1",
        requested_at=DEMO_TIME,
    ),
    withdraw_container.orchestrator.handle(
        (
            f"transferMoney sender_id={source_id} "
            f"receiver_id={withdraw_b.output['user_id']} amount=80.00"
        ),
        idempotency_key="IDEM-CW-2",
        requested_at=DEMO_TIME,
    ),
)
successes = [
    item for item in withdraw_results
    if isinstance(item.output, dict)
    and item.output.get("operation") == "transferMoney"
]
reflections = [
    item for item in withdraw_results
    if item.agent_name == "ReflectionAgent"
]
assert len(successes) == 1 and len(reflections) == 1
assert withdraw_container.snapshot().wallets[source_id].balance == Decimal("20.00")

deposit_container, _ = await isolated_container("concurrent-deposit")
deposit_one = await create_demo_user(
    deposit_container, "Deposit One", 411, "100.00", "CD-1"
)
deposit_two = await create_demo_user(
    deposit_container, "Deposit Two", 412, "100.00", "CD-2"
)
deposit_target = await create_demo_user(
    deposit_container, "Deposit Target", 413, "20.00", "CD-T"
)
deposit_results = await asyncio.gather(
    deposit_container.orchestrator.handle(
        (
            f"transferMoney sender_id={deposit_one.output['user_id']} "
            f"receiver_id={deposit_target.output['user_id']} amount=100.00"
        ),
        idempotency_key="IDEM-CD-X1",
        requested_at=DEMO_TIME,
    ),
    deposit_container.orchestrator.handle(
        (
            f"transferMoney sender_id={deposit_two.output['user_id']} "
            f"receiver_id={deposit_target.output['user_id']} amount=100.00"
        ),
        idempotency_key="IDEM-CD-X2",
        requested_at=DEMO_TIME,
    ),
)
deposit_summary = [
    (
        item.agent_name,
        item.output.get("operation")
        if isinstance(item.output, dict)
        else getattr(item.output, "error_code", type(item.output).__name__),
    )
    for item in deposit_results
]
assert all(
    isinstance(item.output, dict)
    and item.output.get("operation") == "transferMoney"
    for item in deposit_results
), deposit_summary
target_id = deposit_target.output["user_id"]
assert deposit_container.snapshot().wallets[target_id].balance == Decimal("220.00")

opposite_container, _ = await isolated_container("opposite")
opposite_a = await create_demo_user(
    opposite_container, "Opposite A", 421, "100.00", "CO-A"
)
opposite_b = await create_demo_user(
    opposite_container, "Opposite B", 422, "100.00", "CO-B"
)
opposite_results = await asyncio.wait_for(
    asyncio.gather(
        opposite_container.orchestrator.handle(
            (
                f"transferMoney sender_id={opposite_a.output['user_id']} "
                f"receiver_id={opposite_b.output['user_id']} amount=10.00"
            ),
            idempotency_key="IDEM-CO-1",
            requested_at=DEMO_TIME,
        ),
        opposite_container.orchestrator.handle(
            (
                f"transferMoney sender_id={opposite_b.output['user_id']} "
                f"receiver_id={opposite_a.output['user_id']} amount=10.00"
            ),
            idempotency_key="IDEM-CO-2",
            requested_at=DEMO_TIME,
        ),
    ),
    timeout=2.0,
)
assert len(opposite_results) == 2
CONCURRENCY_PASSED = True
print("Double-spending final balance: 20.00")
print("Lost-update final target balance: 220.00")
print("Opposite transfers timeout: PASS")
print("PASS — concurrency safety demonstrations")


Double-spending final balance: 20.00
Lost-update final target balance: 220.00
Opposite transfers timeout: PASS
PASS — concurrency safety demonstrations


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 15. OpenAI Agents SDK אופציונלי


        שכבת ה־SDK היא אופציונלית.
        הנתב יכול להשתמש בפלט מובנה, ושלושה handoffs לקריאה בלבד זמינים לבדיקת הונאה,
        אבטחה והסבר.

        בהרצה הרגילה של המחברת לא נוצר provider client ולא נשלחת בקשת רשת.
        אין כלי SDK שמבצע פעולה כספית.

</div>

In [16]:
from agents import AgentOutputSchema
from agentic_payments.infrastructure.llm.sdk_agents import (
    _router_agent,
    _specialist_agents,
)

sdk_router = _router_agent("not-executed")
sdk_specialists = _specialist_agents("not-executed")
sdk_tools = [
    tool.name
    for agent in (
        sdk_specialists.fraud,
        sdk_specialists.security,
        sdk_specialists.explanation,
    )
    for tool in agent.tools
]
assert isinstance(sdk_router.output_type, AgentOutputSchema)
assert sdk_router.output_type.is_strict_json_schema() is False
assert sdk_tools == [
    "get_fraud_review_facts",
    "get_security_review_facts",
    "get_last_action_facts",
]
assert not any(
    marker in " ".join(sdk_tools)
    for marker in ("transfer", "approve", "create_user")
)
print("Optional SDK router schema: locally validated, non-strict provider schema")
print("Read-only SDK tools:", ", ".join(sdk_tools))
print("Financial function tool present: no")
print("Live provider request made: no")


Optional SDK router schema: locally validated, non-strict provider schema
Read-only SDK tools: get_fraud_review_facts, get_security_review_facts, get_last_action_facts
Financial function tool present: no
Live provider request made: no


<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 16. מגבלות


        - זו מערכת לימודית ואינה פלטפורמה בנקאית.
        - קובצי JSON ונעילות asyncio מגינים על תהליך ו־event loop יחידים.
        - מערכת מרובת תהליכים דורשת מסד נתונים טרנזקציוני.
        - Audit Log אינו מנגנון replay.
        - פלט LLM הוא מייעץ; חוקי הכסף נשארים דטרמיניסטיים.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 17. שאלות סיכום


        **מה עבד היטב?**

        גבולות שכבות, `Decimal`, זיכרון עסקי, idempotency ו־Audit Outbox.

        **מדוע נדרשים גם locks וגם idempotency?**

        Locks מגינים מפעולות מקבילות; idempotency מונע ביצוע חוזר של retry זהה.

        **מתי שירות ענן מתאים?**

        כאשר נדרשים מודלים חזקים וזמינות מנוהלת, בכפוף למדיניות פרטיות.

        **מדוע אין צורך במפתח API?**

        הנתב הדטרמיניסטי וכל חוקי התשלום פועלים מקומית.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.7;">

## 18. אימות סופי וניקוי


        האימות האחרון מרכז את תוצאות התרחישים, ההתמדה, המקביליות והבידוד.
        התא שאחריו מנקה את התיקייה הזמנית והוא התא האחרון במחברת.

</div>

In [17]:
final_outbox = await high_restart.flush_outbox()
REPOSITORY_DATA_AFTER = data_snapshot()
FINAL_CHECKS = {
    "repository source imported": (SOURCE_ROOT / "agentic_payments").is_dir(),
    "all ten lecturer scenarios passed": LECTURER_SCENARIOS_PASSED,
    "persistence restart passed": PERSISTENCE_PASSED,
    "outbox passed": OUTBOX_PASSED and final_outbox.pending_after == 0,
    "concurrency passed": CONCURRENCY_PASSED,
    "no API key used": demo_settings.llm_api_key is None,
    "no provider client": demo_container.llm_runtime is None,
    "repository data untouched": REPOSITORY_DATA_AFTER == REPOSITORY_DATA_BEFORE,
}
assert all(FINAL_CHECKS.values())
for label, passed in FINAL_CHECKS.items():
    print(f"PASS — {label}: {'yes' if passed else 'no'}")
print("FINAL NOTEBOOK VALIDATION PASSED")


PASS — repository source imported: yes
PASS — all ten lecturer scenarios passed: yes
PASS — persistence restart passed: yes
PASS — outbox passed: yes
PASS — concurrency passed: yes
PASS — no API key used: yes
PASS — no provider client: yes
PASS — repository data untouched: yes
FINAL NOTEBOOK VALIDATION PASSED


In [18]:
NOTEBOOK_RUNTIME.cleanup()
assert not RUNTIME_ROOT.exists()
print("Temporary notebook runtime cleaned successfully.")


Temporary notebook runtime cleaned successfully.
